# Automobile Dataset Analysis

Exploratory analysis, PCA, and regression on a dataset of vehicles (consumption, power,
weight, origin). All logic lives in `src/` (see the README) — this notebook loads the
data, runs the analysis functions, and renders the charts.

## 1. Load and clean data

In [ ]:
from src.data import load_data, clean_data, QUANTITATIVE_VARS

df = load_data("automobiles.csv")
print(f"Dimensions: {df.shape[0]} rows x {df.shape[1]} columns")
print(f"Variables: {list(df.columns)}")

df_clean = clean_data(df)
print(f"Cleaned dataset: {len(df_clean)} rows")
df_clean.describe()

## 2. Exploratory visualizations

In [ ]:
from src.visualize import plot_distributions

plot_distributions(df_clean)

In [ ]:
from src.visualize import plot_origin_comparison

print(df_clean.groupby('origin_name')[QUANTITATIVE_VARS[:-1]].mean().round(2))
plot_origin_comparison(df_clean)

In [ ]:
from src.visualize import plot_correlation_heatmap

plot_correlation_heatmap(df_clean)

In [ ]:
from src.visualize import plot_pairplot

plot_pairplot(df_clean)

## 3. CAFE compliance analysis (24 mpg threshold)

In [ ]:
from src.visualize import plot_cafe_analysis

n_meets = df_clean['meets_cafe'].sum()
n_total = len(df_clean)
print(f"Cars >=24 mpg: {n_meets}/{n_total} ({n_meets / n_total * 100:.1f}%)")

comparison = df_clean.groupby('meets_cafe')[QUANTITATIVE_VARS].mean().T
comparison.columns = ['< 24 mpg', '>= 24 mpg']
comparison['Diff %'] = ((comparison['>= 24 mpg'] - comparison['< 24 mpg']) / comparison['< 24 mpg'] * 100).round(1)
display(comparison.round(2))

plot_cafe_analysis(df_clean)

## 4. Principal Component Analysis

In [ ]:
from src.pca_analysis import run_pca
from src.visualize import plot_pca_scree, plot_pca_projection, plot_pca_contributions

pca_result = run_pca(df_clean)
print(f"{pca_result.n_90} components needed for 90% variance")

plot_pca_scree(pca_result)
plot_pca_projection(df_clean, pca_result)
plot_pca_contributions(pca_result)

## 5. Predicting MPG: polynomial vs. multiple regression

In [ ]:
from src.regression import polynomial_regression_comparison, multiple_regression
from src.visualize import plot_regression_curves

poly_result = polynomial_regression_comparison(df_clean, degrees=(1, 2, 3))
print(poly_result.results_df)
plot_regression_curves(poly_result)

In [ ]:
multi_result = multiple_regression(
    df_clean,
    features=['cylinders', 'displacement', 'horsepower', 'weight_lbs', 'acceleration'],
)

linear_row = poly_result.results_df.set_index('Degree').loc[1]
print(f"Linear (weight only):  R^2 = {linear_row['R^2 Test']:.4f}, RMSE = {linear_row['RMSE']:.2f}")
print(f"Polynomial degree 2:   R^2 = {poly_result.results_df.set_index('Degree').loc[2, 'R^2 Test']:.4f}, "
      f"RMSE = {poly_result.results_df.set_index('Degree').loc[2, 'RMSE']:.2f}")
print(f"Polynomial degree 3:   R^2 = {poly_result.results_df.set_index('Degree').loc[3, 'R^2 Test']:.4f}, "
      f"RMSE = {poly_result.results_df.set_index('Degree').loc[3, 'RMSE']:.2f}")
print(f"Multiple (5 vars):     R^2 = {multi_result.r2:.4f}, RMSE = {multi_result.rmse:.2f}")